## Libaries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

from statsmodels.stats.anova import AnovaRM
from scipy.stats import ttest_rel
from itertools import combinations
from statsmodels.stats.multitest import multipletests

current_dir = Path().resolve().parent.parent
sys.path.append(str(current_dir))

from src.utilities.load_and_visualize_data import DataAnalysis      # noqa: F402

## Data to analyse

In [14]:
#=====================================#
# Subject-dependent EEG across models #
#=====================================#
subj_EEG_acc = {
    
"LSTM":{
    "EEG":[30.56, 33.33, 40.74, 33.33, 29.63, 34.85, 41.33, 64.20, 33.33, 33.33, 33.33, 33.33, 33.33, 35.90, 33.33, 30.77, 39.13]      
},

"CNN+LSTM":{
    "EEG":[70.83, 64.29, 61.73, 33.33, 35.80, 33.33, 45.33, 77.78, 33.33, 49.38, 52.38, 50.00, 46.15, 37.18, 26.39, 37.18, 47.83],
},

"CNN+LSTM+Attention":{
    "EEG":[77.78, 63.10, 62.96, 53.33, 50.62, 50.00, 58.67, 80.25, 44.87, 60.49, 57.14, 56.41, 50.00, 58.97, 45.83, 43.59, 50.72]
}}

print(np.mean(subj_EEG_acc["LSTM"]['EEG']), np.std(subj_EEG_acc["LSTM"]['EEG']))
print(np.mean(subj_EEG_acc["CNN+LSTM"]['EEG']), np.std(subj_EEG_acc["CNN+LSTM"]['EEG']))
print(np.mean(subj_EEG_acc["CNN+LSTM+Attention"]['EEG']), np.std(subj_EEG_acc["CNN+LSTM+Attention"]['EEG']))

36.10294117647058 7.714083197942261
47.190588235294115 14.104795792916025
56.748823529411766 10.019372929980928


## Analysis

### Subject-dependent across models for EEG

In [15]:
#======================#
# Convert to dataframe #
#======================#
models = list(subj_EEG_acc.keys())
data = np.array([subj_EEG_acc[m]['EEG'] for m in models]).T  # shape: (subjects, models)

df = pd.DataFrame(data, columns=models)
df["subject"] = range(len(df))

# Long format (required for ANOVA)
df_long = df.melt(id_vars=["subject"], var_name="model", value_name="accuracy")

#==========#
# RM-ANOVA #
#==========#
anova = AnovaRM(df_long, depvar="accuracy", subject="subject", within=["model"])
res = anova.fit()

print("\n=== RM-ANOVA ===")
print(res)

#=========================#
# Post-hoc paired t-tests #
#=========================#
print("\n=== Post-hoc (paired t-tests, Holm corrected) ===")

p_vals = []
pairs = []

for m1, m2 in combinations(models, 2):
    t_stat, p = ttest_rel(df[m1], df[m2])
    p_vals.append(p)                        # p-value between models
    pairs.append((m1, m2))                  # Model pair

reject, p_corrected, _, _ = multipletests(p_vals, method='holm')

for i, (m1, m2) in enumerate(pairs):
    print(f"{m1} vs {m2}: p={p_corrected[i]:.4f}, significant={reject[i]}")



=== RM-ANOVA ===
               Anova
      F Value Num DF  Den DF Pr > F
-----------------------------------
model 39.1941 2.0000 32.0000 0.0000


=== Post-hoc (paired t-tests, Holm corrected) ===
LSTM vs CNN+LSTM: p=0.0018, significant=True
LSTM vs CNN+LSTM+Attention: p=0.0000, significant=True
CNN+LSTM vs CNN+LSTM+Attention: p=0.0001, significant=True


### EEG band/regions analysis

In [2]:
#==============================#
# Load dataset and compute PSD #
#==============================#
base_dir = Path().resolve().parents[0] / 'experiment/data'
print(base_dir)
subjects = [f'subject_{nr}' for nr in range(17)]

PSD_ins = DataAnalysis(fs = 125, base_dir = base_dir)
data = PSD_ins.inspect_frequency_ranges(subjects = subjects)

#======================================================================#
# Convert to matrices (subjects × classes × regions × bands) for ANOVA #
#======================================================================#
n_subjects = len(data)
n_classes = len(data[0])
REGIONS = ['prefrontal', 'frontal', 'central', 'temporal', 'parietal']
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']

all_mats = []

for subj_data in data:
    subj_mats = []

    for feat_dict in subj_data:                      # Extract PSD features (per channel x per band) for Class1 and then class2
        
        # Convert dict → matrix (channels × bands)
        mat = np.zeros((len(REGIONS), len(BANDS)))

        for i, ch in enumerate(REGIONS):
            for j, band in enumerate(BANDS):
                mat[i, j] = np.mean(feat_dict[ch][band])
        
        subj_mats.append(mat)
    
    all_mats.append(subj_mats)

all_mats = np.array(all_mats)           # Shape: (Subj, class, region, band)

S, C, R, B = all_mats.shape

C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_index_finger_2026-02-05 14-19-21.csv
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_index_finger_2026-02-05 14-24-42.csv
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_index_finger_2026-02-05 14-37-01.csv

=====FUNC : reject_routine =====

Final combined bad epoch indicies: [15 21 29 31 33 41 44 46 50 51 52 61 62 64 79 83]
Total bad epochs = 16 out of 90
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_thumb_finger_2026-02-05 14-46-00.csv
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_thumb_finger_2026-02-05 14-51-56.csv
C:\Users\Bruger\Documents\hybrid_bci\src\experiment\data\subject_0\EEG\flex_thumb_finger_2026-02-05 14-57-32.csv

=====FUNC : reject_routine =====

Final combined bad epoch indicies: [13 14 25 34 37 42 53 59 61 78 79 81 82 83]

In [ ]:
#========================#
# Convert to long format #
#========================#
rows = []

for s in range(n_subjects):
    for c in range(n_classes):
        for r in range(len(REGIONS)):
            for b in range(len(BANDS)):
                rows.append({
                    "subject": s,
                    "class": f"class_{c}",
                    "region": f"{REGIONS[r]}",
                    "band": f"{BANDS[b]}",
                    "condition": f"C{c}_R{r}_B{b}",
                    "psd": all_mats[s, c, r, b]
                })

df_long = pd.DataFrame(rows)

anova = AnovaRM(
    df_long,
    depvar="psd",
    subject="subject",
    within=["condition"]
)

res = anova.fit()
print(res)


#=========================#
# Post-hoc paired t-tests #
#=========================#
print("\n=== Post-hoc (paired t-tests, Holm corrected) ===")
# [rest, contract, release] -> [class 0, class 1, class 2]

class_pairs = list(combinations(range(n_classes), 2))

for c1, c2 in class_pairs:
    print(f"\n=== Comparing class {c1} vs {c2} ===")

    p_vals = []
    labels = []

    for r in range(R):
        for b in range(B):

            x = all_mats[:, c1, r, b]
            y = all_mats[:, c2, r, b]

            _, p = ttest_rel(x, y)

            p_vals.append(p)
            labels.append((r, b))

    reject, p_corr, _, _ = multipletests(p_vals, method='holm')

    for i, (r, b) in enumerate(labels):
        if reject[i]:
            print(f"{REGIONS[r]}, {BANDS[b]}: p={p_corr[i]:.4f}")

#===========#
# Cohen's d #
#===========#
print("\n=== Cohen's d for significant differences ===")
cohend = PSD_ins.compute_multiclass_separability(data_classes = data, REGIONS = REGIONS, BANDS = BANDS)
print(cohend)

                  Anova
          F Value  Num DF   Den DF  Pr > F
------------------------------------------
condition 20.2596 74.0000 1184.0000 0.0000


=== Post-hoc (paired t-tests, Holm corrected) ===

=== Comparing class 0 vs 1 ===
prefrontal, beta: p=0.0432
frontal, beta: p=0.0432
central, delta: p=0.0067
central, beta: p=0.0275

=== Comparing class 0 vs 2 ===
prefrontal, delta: p=0.0263
frontal, delta: p=0.0179
frontal, beta: p=0.0418
central, delta: p=0.0002
central, alpha: p=0.0475
central, beta: p=0.0084
temporal, delta: p=0.0047

=== Comparing class 1 vs 2 ===
prefrontal, alpha: p=0.0474
temporal, delta: p=0.0481
temporal, alpha: p=0.0481

=== Cohen's d for significant differences ===
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
(array([[[0.15133235, 0.12786674, 0.19934493, 0.27240378, 0.14243232],
        [0.2360488 , 0.14415202, 0.22232897, 0.37011325, 0.13737762],
        [0.34706121, 0.13892488, 0.24873549, 0.46175287, 0.17026009],
        [0.16190411, 0.10442746, 0.1823653 , 0.235